# Notebook 01 — EDA SIHSUS SP 2025
**Projeto:** Análise e Predição de Custos de Internação no SIHSUS (SP/2025)  
**Objetivo:** Extrair, enriquecer e explorar os dados de internações hospitalares do estado de São Paulo.

---
### Decisão de arquitetura
A biblioteca `datasus-etl` já executa internamente as etapas de:
- **Download** do FTP do DATASUS (SOR)
- **Conversão** DBC → DBF → DuckDB (SOT)
- **Enriquecimento** com IBGE (municípios) e CID-10 (SPEC)

Portanto, a separação explícita SOR/SOT é desnecessária: o pipeline entrega diretamente a camada **SPEC** (parquet particionado, enriquecido e pronto para modelagem). O `spec_sih_2025_sp.parquet` é gerado ao final deste notebook.

## 1. Setup — Instalação e Importações

In [ ]:
# Instalar dependências (execute apenas na primeira vez)
import subprocess, sys

packages = [
    'datasus-etl', 'pandas', 'numpy', 'pyarrow',
    'matplotlib', 'seaborn', 'polars', 'duckdb', 'scikit-learn'
]
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '--quiet'] + packages)
print('✅ Instalação concluída.')

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import os
import pandas as pd
import numpy as np
import polars as pl
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns

# Estilo visual
sns.set_theme(style='whitegrid', palette='muted', font_scale=1.1)
plt.rcParams['figure.dpi'] = 120
plt.rcParams['figure.figsize'] = (12, 5)

# Diretórios do projeto
DATA_DIR   = './data'
OUTPUT_DIR = './outputs'
os.makedirs(DATA_DIR, exist_ok=True)
os.makedirs(OUTPUT_DIR, exist_ok=True)

print('✅ Pacotes importados.')

---
## 2. Extração via datasus-etl (SOR → SOT → SPEC automaticamente)

In [ ]:
from datasus_etl.config import PipelineConfig
from datasus_etl.pipeline.sihsus_pipeline import SihsusPipeline

# Configuração do pipeline
config = PipelineConfig.create(
    source     = 'sihsus',
    start_date = '2025-01-01',
    end_date   = '2025-12-31',
    ufs        = ['SP'],
    data_dir   = DATA_DIR,
)

# Callback de progresso
def on_progress(event):
    pct = f"{event.pct:.0%}" if hasattr(event, 'pct') and event.pct is not None else ''
    print(f"  [{event.stage}] {event.message} {pct}")

print('🚀 Iniciando pipeline SIHSUS SP 2025...')
print('   (Download + conversão DBC→Parquet + enriquecimento IBGE/CID-10)')
print('   Isso pode levar alguns minutos dependendo da conexão.\n')

pipeline = SihsusPipeline(config)
pipeline.run(progress=on_progress)

print('\n✅ Pipeline concluído!')

---
## 3. Carregamento do Parquet gerado pelo ETL

In [ ]:
import glob

# O datasus-etl grava em: data/datasus_db/sihsus/**/*.parquet
PARQUET_GLOB = f'{DATA_DIR}/datasus_db/sihsus/**/*.parquet'

parquet_files = glob.glob(PARQUET_GLOB, recursive=True)
print(f'Arquivos parquet encontrados: {len(parquet_files)}')
for f in parquet_files[:10]:
    print(' ', f)

if not parquet_files:
    raise FileNotFoundError(
        f'Nenhum parquet encontrado em {PARQUET_GLOB}.\n'
        'Verifique se o pipeline rodou com sucesso e ajuste o caminho.'
    )

In [ ]:
# Leitura com Polars (eficiente para parquet particionado) → converte para pandas
df_raw = pl.scan_parquet(PARQUET_GLOB).collect().to_pandas()

print(f'Shape: {df_raw.shape}')
print(f'Colunas ({len(df_raw.columns)}):')
print(df_raw.columns.tolist())

In [ ]:
df_raw.head(3)

---
## 4. Padronização e construção da camada SPEC

In [ ]:
# ──────────────────────────────────────────────────────────────────
# 4.1 Identificação da coluna alvo
# O SIHSUS usa VAL_TOT como valor total da internação.
# Vamos normalizar para 'valor_total_internacao'.
# ──────────────────────────────────────────────────────────────────

# Mapeamento flexível: identifica a coluna de valor total
VALOR_COL_CANDIDATES = ['VAL_TOT', 'VALOR_TOTAL', 'valor_total_internacao', 'val_tot']
valor_col_original = None
for c in VALOR_COL_CANDIDATES:
    if c in df_raw.columns:
        valor_col_original = c
        break

if valor_col_original is None:
    print('⚠️  Coluna de valor total não encontrada. Colunas numéricas disponíveis:')
    print(df_raw.select_dtypes(include='number').columns.tolist())
    raise ValueError('Ajuste VALOR_COL_CANDIDATES para o nome correto da coluna de valor.')

print(f'✅ Coluna de valor total identificada: {valor_col_original}')

In [ ]:
# ──────────────────────────────────────────────────────────────────
# 4.2 Cópia e renomeação padronizada
# ──────────────────────────────────────────────────────────────────
df = df_raw.copy()

# Padronizar nomes para lowercase
df.columns = [c.lower() for c in df.columns]
valor_col_original = valor_col_original.lower()

# Renomear alvo
if valor_col_original != 'valor_total_internacao':
    df.rename(columns={valor_col_original: 'valor_total_internacao'}, inplace=True)

print('Colunas após normalização:')
print(df.columns.tolist())

In [ ]:
# ──────────────────────────────────────────────────────────────────
# 4.3 Tipagem de datas
# datasus-etl já parseia DT_INTER e DT_SAIDA como DATE.
# Caso venham como string, forçamos a conversão aqui.
# ──────────────────────────────────────────────────────────────────
DATE_COLS = ['dt_inter', 'dt_saida']
for col in DATE_COLS:
    if col in df.columns:
        df[col] = pd.to_datetime(df[col], errors='coerce')
        print(f'  {col}: {df[col].dtype}')

In [ ]:
# ──────────────────────────────────────────────────────────────────
# 4.4 Variáveis derivadas
# ──────────────────────────────────────────────────────────────────

# Tempo de internação (dias)
if 'dt_inter' in df.columns and 'dt_saida' in df.columns:
    df['tempo_internacao'] = (df['dt_saida'] - df['dt_inter']).dt.days
    df['tempo_internacao'] = df['tempo_internacao'].clip(lower=0)
elif 'dias_perm' in df.columns:
    df['tempo_internacao'] = pd.to_numeric(df['dias_perm'], errors='coerce')
else:
    df['tempo_internacao'] = np.nan
    print('⚠️  Não foi possível calcular tempo_internacao.')

# Mês e ano de internação
if 'dt_inter' in df.columns:
    df['mes_internacao'] = df['dt_inter'].dt.month
    df['ano_internacao'] = df['dt_inter'].dt.year
elif 'competencia' in df.columns:
    competencia_str = df['competencia'].astype(str)
    df['ano_internacao'] = competencia_str.str[:4].astype(int, errors='ignore')
    df['mes_internacao'] = competencia_str.str[4:6].astype(int, errors='ignore')

# Faixa etária
IDADE_COL = next((c for c in df.columns if 'idade' in c), None)
if IDADE_COL:
    df[IDADE_COL] = pd.to_numeric(df[IDADE_COL], errors='coerce')
    bins   = [0, 1, 5, 12, 18, 29, 39, 49, 59, 69, 79, 999]
    labels = ['<1', '1-4', '5-11', '12-17', '18-28', '29-38',
              '39-48', '49-58', '59-68', '69-78', '79+']
    df['faixa_etaria'] = pd.cut(df[IDADE_COL], bins=bins, labels=labels, right=True)
    print(f'  Faixa etária criada a partir de: {IDADE_COL}')

print('\n✅ Variáveis derivadas criadas:')
print('  - tempo_internacao, mes_internacao, ano_internacao, faixa_etaria')

In [ ]:
# ──────────────────────────────────────────────────────────────────
# 4.5 Verificação de faltantes e duplicatas
# ──────────────────────────────────────────────────────────────────
print('=== Faltantes por coluna (%) ===')
missing = (df.isnull().mean() * 100).round(2)
missing_relevant = missing[missing > 0].sort_values(ascending=False)
print(missing_relevant.to_string() if len(missing_relevant) > 0 else '  Nenhum faltante!')

print(f'\n=== Duplicatas ===')
n_dup = df.duplicated().sum()
print(f'  Linhas duplicadas: {n_dup} ({n_dup/len(df)*100:.2f}%)')
if n_dup > 0:
    df = df.drop_duplicates()
    print(f'  → Removidas. Shape final: {df.shape}')

In [ ]:
# ──────────────────────────────────────────────────────────────────
# 4.6 Filtragem da variável alvo
# ──────────────────────────────────────────────────────────────────
df['valor_total_internacao'] = pd.to_numeric(df['valor_total_internacao'], errors='coerce')

# Remover registros sem valor ou com valor <= 0
n_antes = len(df)
df = df[df['valor_total_internacao'].notna() & (df['valor_total_internacao'] > 0)]
print(f'Registros removidos (valor inválido): {n_antes - len(df):,}')
print(f'Shape após limpeza: {df.shape}')

In [ ]:
# ──────────────────────────────────────────────────────────────────
# 4.7 Verificação do enriquecimento CID-10 e Municípios
# O datasus-etl já enriquece automaticamente.
# Aqui confirmamos quais colunas de enriquecimento estão presentes.
# ──────────────────────────────────────────────────────────────────
print('=== Colunas de enriquecimento disponíveis ===')

# CID-10
cid_cols = [c for c in df.columns if 'cid' in c.lower() or 'diag' in c.lower()]
print(f'  CID / Diagnóstico : {cid_cols}')

# Município
mun_cols = [c for c in df.columns if 'mun' in c.lower() or 'ibge' in c.lower() or 'municipio' in c.lower()]
print(f'  Município / IBGE  : {mun_cols}')

# UF
uf_cols = [c for c in df.columns if c.lower() in ['uf', 'sg_uf', 'uf_zipo', 'uf_inter']]
print(f'  UF                : {uf_cols}')

# Especialidade
esp_cols = [c for c in df.columns if 'espec' in c.lower() or 'especialidade' in c.lower()]
print(f'  Especialidade     : {esp_cols}')

---
## 5. EDA — Análise Exploratória

In [ ]:
# ──────────────────────────────────────────────────────────────────
# 5.1 Estatísticas descritivas gerais
# ──────────────────────────────────────────────────────────────────
print('=== Estatísticas Descritivas — valor_total_internacao ===')
desc = df['valor_total_internacao'].describe(percentiles=[.25, .5, .75, .90, .95, .99])
print(desc.apply(lambda x: f'R$ {x:,.2f}' if x > 1 else f'{x:.4f}'))

skew = df['valor_total_internacao'].skew()
kurt = df['valor_total_internacao'].kurt()
print(f'\nAssimetria (skewness): {skew:.2f}')
print(f'Curtose:               {kurt:.2f}')

In [ ]:
# ──────────────────────────────────────────────────────────────────
# 5.2 Distribuição do valor total (histograma + boxplot + log)
# ──────────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Histograma escala original
axes[0].hist(df['valor_total_internacao'], bins=80, color='steelblue', edgecolor='white', linewidth=0.3)
axes[0].set_title('Distribuição — Escala Original')
axes[0].set_xlabel('Valor Total (R$)')
axes[0].set_ylabel('Frequência')
axes[0].xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'R${x/1000:.0f}k'))

# Histograma escala log
log_vals = np.log1p(df['valor_total_internacao'])
axes[1].hist(log_vals, bins=80, color='teal', edgecolor='white', linewidth=0.3)
axes[1].set_title('Distribuição — Escala Log(1+x)')
axes[1].set_xlabel('log(1 + Valor Total)')
axes[1].set_ylabel('Frequência')

# Boxplot
axes[2].boxplot(df['valor_total_internacao'], vert=True, patch_artist=True,
                boxprops=dict(facecolor='steelblue', alpha=0.7),
                medianprops=dict(color='red', linewidth=2))
axes[2].set_title('Boxplot — Valor Total')
axes[2].set_ylabel('Valor Total (R$)')
axes[2].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'R${x/1000:.0f}k'))

plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/01_distribuicao_valor_total.png', bbox_inches='tight')
plt.show()
print(f'💡 Insight: Distribuição fortemente assimétrica à direita (skew={skew:.1f}). '
      'Modelagem em escala log é recomendada.')

In [ ]:
# ──────────────────────────────────────────────────────────────────
# 5.3 Tendência mensal de volume e custo
# ──────────────────────────────────────────────────────────────────
if 'mes_internacao' in df.columns:
    monthly = df.groupby('mes_internacao').agg(
        volume=('valor_total_internacao', 'count'),
        custo_total=('valor_total_internacao', 'sum'),
        custo_medio=('valor_total_internacao', 'mean')
    ).reset_index()
    monthly['mes'] = monthly['mes_internacao'].astype(int)

    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    mes_labels = ['Jan','Fev','Mar','Abr','Mai','Jun','Jul','Ago','Set','Out','Nov','Dez']

    axes[0].bar(monthly['mes'], monthly['volume'], color='steelblue')
    axes[0].set_title('Volume de Internações por Mês')
    axes[0].set_xlabel('Mês'); axes[0].set_ylabel('Nº Internações')
    axes[0].set_xticks(range(1, 13)); axes[0].set_xticklabels(mes_labels, rotation=45)

    axes[1].bar(monthly['mes'], monthly['custo_total'] / 1e6, color='teal')
    axes[1].set_title('Custo Total por Mês (R$ Mi)')
    axes[1].set_xlabel('Mês'); axes[1].set_ylabel('R$ Milhões')
    axes[1].set_xticks(range(1, 13)); axes[1].set_xticklabels(mes_labels, rotation=45)

    axes[2].plot(monthly['mes'], monthly['custo_medio'], marker='o', color='coral', linewidth=2)
    axes[2].set_title('Custo Médio por Internação (Mensal)')
    axes[2].set_xlabel('Mês'); axes[2].set_ylabel('R$ (médio)')
    axes[2].set_xticks(range(1, 13)); axes[2].set_xticklabels(mes_labels, rotation=45)
    axes[2].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'R${x:,.0f}'))

    plt.tight_layout()
    plt.savefig(f'{OUTPUT_DIR}/02_tendencia_mensal.png', bbox_inches='tight')
    plt.show()
else:
    print('⚠️  Coluna mes_internacao não disponível para análise mensal.')

In [ ]:
# ──────────────────────────────────────────────────────────────────
# 5.4 Custo por Especialidade
# ──────────────────────────────────────────────────────────────────
ESPECIALIDADE_COL = next((c for c in df.columns if 'espec' in c.lower()), None)

if ESPECIALIDADE_COL:
    esp_stats = (
        df.groupby(ESPECIALIDADE_COL)['valor_total_internacao']
        .agg(['count', 'mean', 'sum'])
        .rename(columns={'count': 'volume', 'mean': 'custo_medio', 'sum': 'custo_total'})
        .sort_values('custo_total', ascending=False)
        .head(15)
    )

    fig, axes = plt.subplots(1, 2, figsize=(18, 6))

    # Custo total por especialidade
    axes[0].barh(esp_stats.index.astype(str)[::-1],
                  esp_stats['custo_total'][::-1] / 1e6, color='steelblue')
    axes[0].set_title('Top 15 Especialidades — Custo Total (R$ Mi)')
    axes[0].set_xlabel('R$ Milhões')

    # Custo médio por especialidade
    axes[1].barh(esp_stats.index.astype(str)[::-1],
                  esp_stats['custo_medio'][::-1], color='coral')
    axes[1].set_title('Top 15 Especialidades — Custo Médio (R$)')
    axes[1].set_xlabel('R$ (médio por internação)')

    plt.tight_layout()
    plt.savefig(f'{OUTPUT_DIR}/03_custo_especialidade.png', bbox_inches='tight')
    plt.show()
    print(esp_stats.to_string())
else:
    print('⚠️  Coluna de especialidade não encontrada. Colunas disponíveis:', df.columns.tolist())

In [ ]:
# ──────────────────────────────────────────────────────────────────
# 5.5 Custo por Sexo
# ──────────────────────────────────────────────────────────────────
SEXO_COL = next((c for c in df.columns if c.lower() in ['sexo', 'ds_sexo', 'sex']), None)

if SEXO_COL:
    sexo_stats = df.groupby(SEXO_COL)['valor_total_internacao'].agg(['count', 'mean', 'median'])
    sexo_stats.columns = ['Volume', 'Custo Médio', 'Custo Mediano']

    fig, axes = plt.subplots(1, 2, figsize=(12, 5))

    axes[0].bar(sexo_stats.index.astype(str), sexo_stats['Volume'], color=['steelblue','coral'])
    axes[0].set_title('Volume de Internações por Sexo')
    axes[0].set_ylabel('Nº Internações')

    axes[1].bar(sexo_stats.index.astype(str), sexo_stats['Custo Médio'], color=['steelblue','coral'])
    axes[1].set_title('Custo Médio por Sexo (R$)')
    axes[1].set_ylabel('R$ (médio por internação)')
    axes[1].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'R${x:,.0f}'))

    plt.tight_layout()
    plt.savefig(f'{OUTPUT_DIR}/04_custo_sexo.png', bbox_inches='tight')
    plt.show()
    print(sexo_stats)
else:
    print('⚠️  Coluna de sexo não encontrada.')

In [ ]:
# ──────────────────────────────────────────────────────────────────
# 5.6 Distribuição por Faixa Etária
# ──────────────────────────────────────────────────────────────────
if 'faixa_etaria' in df.columns:
    fe_stats = df.groupby('faixa_etaria', observed=True)['valor_total_internacao'].agg(['count','mean'])
    fe_stats.columns = ['Volume', 'Custo Médio']

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    axes[0].bar(fe_stats.index.astype(str), fe_stats['Volume'], color='steelblue')
    axes[0].set_title('Volume de Internações por Faixa Etária')
    axes[0].set_xlabel('Faixa Etária'); axes[0].set_ylabel('Nº Internações')
    axes[0].tick_params(axis='x', rotation=45)

    axes[1].bar(fe_stats.index.astype(str), fe_stats['Custo Médio'], color='teal')
    axes[1].set_title('Custo Médio por Faixa Etária (R$)')
    axes[1].set_xlabel('Faixa Etária'); axes[1].set_ylabel('R$ (médio)')
    axes[1].tick_params(axis='x', rotation=45)
    axes[1].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'R${x:,.0f}'))

    plt.tight_layout()
    plt.savefig(f'{OUTPUT_DIR}/05_custo_faixa_etaria.png', bbox_inches='tight')
    plt.show()

In [ ]:
# ──────────────────────────────────────────────────────────────────
# 5.7 Top 10 CID-10 por custo médio
# ──────────────────────────────────────────────────────────────────
CID_COL = next((c for c in df.columns if 'cid' in c.lower() and 'diag' not in c.lower()), None)
if not CID_COL:
    CID_COL = next((c for c in df.columns if 'diag' in c.lower()), None)

if CID_COL:
    cid_stats = (
        df.groupby(CID_COL)['valor_total_internacao']
        .agg(['count','mean'])
        .query('count >= 100')  # mínimo de 100 registros
        .sort_values('mean', ascending=False)
        .head(10)
    )

    fig, ax = plt.subplots(figsize=(12, 6))
    ax.barh(cid_stats.index.astype(str)[::-1], cid_stats['mean'][::-1], color='mediumpurple')
    ax.set_title('Top 10 CID-10 — Maior Custo Médio (min. 100 internações)')
    ax.set_xlabel('Custo Médio (R$)')
    ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'R${x:,.0f}'))
    plt.tight_layout()
    plt.savefig(f'{OUTPUT_DIR}/06_top_cid_custo_medio.png', bbox_inches='tight')
    plt.show()
    print(cid_stats)
else:
    print('⚠️  Coluna CID não encontrada. Colunas disponíveis:', df.columns.tolist())

In [ ]:
# ──────────────────────────────────────────────────────────────────
# 5.8 Top 15 Municípios — Custo Total
# ──────────────────────────────────────────────────────────────────
MUN_COL = next((c for c in df.columns if 'municipio' in c.lower() or 'mun_res' in c.lower()
                or 'nome_municipio' in c.lower() or 'nm_municipio' in c.lower()), None)
if not MUN_COL:
    # Fallback para código numérico
    MUN_COL = next((c for c in df.columns if 'mun' in c.lower()), None)

if MUN_COL:
    mun_stats = (
        df.groupby(MUN_COL)['valor_total_internacao']
        .agg(['count','sum','mean'])
        .sort_values('sum', ascending=False)
        .head(15)
    )
    mun_stats.columns = ['Volume', 'Custo Total', 'Custo Médio']

    fig, ax = plt.subplots(figsize=(12, 7))
    ax.barh(mun_stats.index.astype(str)[::-1], mun_stats['Custo Total'][::-1] / 1e6, color='seagreen')
    ax.set_title('Top 15 Municípios — Custo Total de Internações (R$ Mi)')
    ax.set_xlabel('R$ Milhões')
    plt.tight_layout()
    plt.savefig(f'{OUTPUT_DIR}/07_top_municipios.png', bbox_inches='tight')
    plt.show()
else:
    print('⚠️  Coluna de município não encontrada.')

In [ ]:
# ──────────────────────────────────────────────────────────────────
# 5.9 Análise de outliers via IQR
# ──────────────────────────────────────────────────────────────────
Q1 = df['valor_total_internacao'].quantile(0.25)
Q3 = df['valor_total_internacao'].quantile(0.75)
IQR = Q3 - Q1
outlier_mask = (df['valor_total_internacao'] < Q1 - 3*IQR) | (df['valor_total_internacao'] > Q3 + 3*IQR)

print(f'=== Análise de Outliers (3×IQR) ===')
print(f'  Q1: R$ {Q1:,.2f}')
print(f'  Q3: R$ {Q3:,.2f}')
print(f'  IQR: R$ {IQR:,.2f}')
print(f'  Outliers detectados: {outlier_mask.sum():,} ({outlier_mask.mean()*100:.2f}%)')
print(f'  Valor máximo outlier: R$ {df.loc[outlier_mask, "valor_total_internacao"].max():,.2f}')

---
## 6. Salvar camada SPEC

In [ ]:
# ──────────────────────────────────────────────────────────────────
# Seleção das colunas relevantes para modelagem
# Removemos colunas de identificação de paciente (anonimização)
# e colunas com > 70% de faltantes
# ──────────────────────────────────────────────────────────────────
COLS_REMOVER_ETICA = [  # Atributos que não devem entrar no modelo por privacidade
    'n_aih', 'cpf_aut', 'cpf', 'naih', 'sequencia',
]

df_spec = df.copy()

# Remover colunas de identificação
for c in COLS_REMOVER_ETICA:
    if c in df_spec.columns:
        df_spec.drop(columns=[c], inplace=True)
        print(f'  Removida por privacidade: {c}')

# Remover colunas com > 70% faltantes
missing_rate = df_spec.isnull().mean()
cols_muitos_faltantes = missing_rate[missing_rate > 0.70].index.tolist()
if cols_muitos_faltantes:
    print(f'  Removidas por > 70% faltantes: {cols_muitos_faltantes}')
    df_spec.drop(columns=cols_muitos_faltantes, inplace=True)

print(f'\nShape final SPEC: {df_spec.shape}')
print(f'Colunas finais: {df_spec.columns.tolist()}')

In [ ]:
# Salvar SPEC
SPEC_PATH = './spec_sih_2025_sp.parquet'
df_spec.to_parquet(SPEC_PATH, index=False)
print(f'✅ spec_sih_2025_sp.parquet salvo em: {SPEC_PATH}')
print(f'   Shape: {df_spec.shape}')
print(f'   Tamanho: {os.path.getsize(SPEC_PATH) / 1024 / 1024:.1f} MB')

---
## 7. Relatório EDA — Resumo de Insights

Execute a célula abaixo para gerar um resumo automático.

In [ ]:
print('='*60)
print('  RELATÓRIO EDA — SIHSUS SP 2025')
print('='*60)
print(f"\n📊 Total de internações analisadas : {len(df_spec):,}")
print(f"💰 Custo total (R$)                : R$ {df_spec['valor_total_internacao'].sum():,.2f}")
print(f"💰 Custo médio por internação      : R$ {df_spec['valor_total_internacao'].mean():,.2f}")
print(f"💰 Custo mediano por internação    : R$ {df_spec['valor_total_internacao'].median():,.2f}")
print(f"⏱  Tempo médio de internação       : {df_spec['tempo_internacao'].mean():.1f} dias" if 'tempo_internacao' in df_spec.columns else '')
print(f"\n🔎 Assimetria (skewness)            : {df_spec['valor_total_internacao'].skew():.2f}")
print(f"   → Recomendado: treinar com log(valor_total_internacao)")
print(f"\n📁 Arquivo SPEC gerado              : spec_sih_2025_sp.parquet")
print(f"📁 Gráficos salvos em               : {OUTPUT_DIR}/")
print('\n✅ EDA concluída com sucesso!')